In [28]:
train_values <- read.csv("train_values.csv",stringsAsFactors = T)
test_values <- read.csv("test_values.csv",stringsAsFactors = T)
train_labels <- read.csv("train_labels.csv",stringsAsFactors = T)
submission_format <- read.csv("submission_format.csv",stringsAsFactors = T)
data <- merge(train_values,train_labels,by=c('building_id','building_id'),all.x=T)
test <- test_values

# Data Pre-procesing 

On dispose d'information sur les variable que l'ordinateur lui na pas. 
On vas essayer de lui les transmettre.

## Discution des package uttiliser 

In [ ]:
#install.packages("ggplot2")
#install.packages("lattice")
#install.packages("caret")
#install.packages("dplyr")

In [5]:
library(ggplot2)
library(lattice)
library(caret)
library(dplyr)
library(tictoc)

## Pertinance des variable et de leur type 

#### damage_grade

Les variable damage_grade que l'on cherche a predire prenne 3 valeur possible sous forme d'entier 1,2,3.
En se sens, il s'agit de variables catégoriques plutôt que de variable numérique. 
En effet, si on avait remplacer damage_grade 3 par damage_grade 10 on aurait pas changer l'information a notre disposition. 
Alors que l'ordinateur traiterait ses information différament. 
Cependnant, puisqu'il existe une structure d'ordre au sens de cette variable (i.e 1 < 2 < 3) cella peux quand même faire du sens de les traitrer comme des variable numérique. 
On pourait evisagabler de changer leur valeur tous en preservant l'odre interne comme un parametre de tunings.

#### building_id 

la variable building_id est un identifiant unique et aléatoir et ne contien pas d'information uttile pour notre problème. 

#### geo_level 

désigne les region géographic dans les quelle le batiment se situe.
Les 3 variable on une strucutre interne qui ne sont pas retranscrite dans le data set (i.e geo_level_3 inclu geo_level_2 inclue geo_level_1)
De plus, il s'agit d'entier qui n'apporte pas en temps que telle d'information. 
Si l'on echange les les indice des zone 1 et 1200 dans geo_level_3, cella ne change pas l'information a la quelle on dispose mais changera nos algorithme. 
On ne devrait pas les traiter comme variable numérique mais plutôt comme variable catégorique. 
Cependant, en pratique, elle prene un trop grand nombre de valeur différente que pour pouvoir s'en servir commme variable catégorique. 
Intuitivement, il s'aggit de variable important elle devrait donc porter une iformation importante.

#### variable catégorique 

L'information des autre varibale catégorique nous étant cacher, on se sera pas en messur d'en tirer plus d'inforamtion. 

In [58]:
length(unique(data$geo_level_3_id))

[1] 11595

In [59]:
length(unique(data$geo_level_2_id))

[1] 1414

In [60]:
length(unique(data$geo_level_1_id))

[1] 31

In [29]:
mean_damage_level_1 <- 0:30
mean_damage_level_2 <- 0:1427
mean_damage_level_3 <- 0:12567

sd_damage_level_1 <- 0:30
sd_damage_level_2 <- 0:1427
sd_damage_level_3 <- 0:12567


mean_level_1 <- data %>% group_by(geo_level_1_id) %>% 
  summarise(mean_damage=mean(damage_grade),
            .groups = 'drop')

mean_level_2 <- data %>% group_by(geo_level_2_id) %>% 
  summarise(mean_damage=mean(damage_grade),
            .groups = 'drop')

mean_level_3 <- data %>% group_by(geo_level_3_id) %>% 
  summarise(mean_damage=mean(damage_grade),
            .groups = 'drop')

sd_level_1 <- data %>% group_by(geo_level_1_id) %>% 
  summarise(sd_damage=sd(damage_grade),
            .groups = 'drop')

sd_level_2 <- data %>% group_by(geo_level_2_id) %>% 
  summarise(sd_damage=sd(damage_grade),
            .groups = 'drop')

sd_level_3 <- data %>% group_by(geo_level_3_id) %>% 
  summarise(sd_damage=sd(damage_grade),
            .groups = 'drop')


for (i in 1:31){mean_damage_level_1[i] <- mean_level_1[i,2]}  

k <- 1
for (i in 1:1428){if(i-1 == mean_level_2[k,1]){ 
        mean_damage_level_2[i] <- mean_level_2[k,2]
        k <- k+1 } else { mean_damage_level_2[i] <- NA}}

k <- 1
for (i in 1:12568){if(i-1 == mean_level_3[k,1]){ 
        mean_damage_level_3[i] <- mean_level_3[k,2]
        k <- k+1 } else { mean_damage_level_3[i] <- NA}}


for (i in 1:31){sd_damage_level_1[i] <- sd_level_1[i,2]}  

k <- 1
for (i in 1:1428){if(i-1 == sd_level_2[k,1]){ 
        sd_damage_level_2[i] <- sd_level_2[k,2]
        k <- k+1 } else { sd_damage_level_2[i] <- NA}}

k <- 1
for (i in 1:12568){if(i-1 == sd_level_3[k,1]){ 
        sd_damage_level_3[i] <- sd_level_3[k,2]
        k <- k+1 } else { sd_damage_level_3[i] <- NA}}

In [ ]:
for (i in 1:nrow(data)){
    if(is.na(mean_damage_level_3[data[i,c("geo_level_3_id")]+1])){
        if(is.na(mean_damage_level_2[data[i,c("geo_level_2_id")]+1])){
            data$geo_level_3_mean_damage[i] <- as.numeric(unlist(mean_damage_level_1[data[i,c("geo_level_1_id")]+1]))
            data$geo_level_2_mean_damage[i] <- as.numeric(unlist(mean_damage_level_1[data[i,c("geo_level_1_id")]+1]))
            data$geo_level_1_mean_damage[i] <- as.numeric(unlist(mean_damage_level_1[data[i,c("geo_level_1_id")]+1]))
        } else {data$geo_level_3_mean_damage[i] <- as.numeric(unlist(mean_damage_level_2[data[i,c("geo_level_2_id")]+1]))
                data$geo_level_2_mean_damage[i] <- as.numeric(unlist(mean_damage_level_2[data[i,c("geo_level_2_id")]+1]))
                data$geo_level_1_mean_damage[i] <- as.numeric(unlist(mean_damage_level_1[data[i,c("geo_level_1_id")]+1]))}
    } else {data$geo_level_3_mean_damage[i] <- as.numeric(unlist(mean_damage_level_3[data[i,c("geo_level_3_id")]+1]))
            data$geo_level_2_mean_damage[i] <- as.numeric(unlist(mean_damage_level_2[data[i,c("geo_level_2_id")]+1]))
            data$geo_level_1_mean_damage[i] <- as.numeric(unlist(mean_damage_level_1[data[i,c("geo_level_1_id")]+1]))}
}

for (i in 1:nrow(data)){
    if(is.na(sd_damage_level_3[data[i,c("geo_level_3_id")]+1])){
        if(is.na(sd_damage_level_2[data[i,c("geo_level_2_id")]+1])){
            data$geo_level_3_sd_damage[i] <- as.numeric(unlist(sd_damage_level_1[data[i,c("geo_level_1_id")]+1]))
            data$geo_level_2_sd_damage[i] <- as.numeric(unlist(sd_damage_level_1[data[i,c("geo_level_1_id")]+1]))
            data$geo_level_1_sd_damage[i] <- as.numeric(unlist(sd_damage_level_1[data[i,c("geo_level_1_id")]+1]))
        } else {data$geo_level_3_sd_damage[i] <- as.numeric(unlist(sd_damage_level_2[data[i,c("geo_level_2_id")]+1]))
                data$geo_level_2_sd_damage[i] <- as.numeric(unlist(sd_damage_level_2[data[i,c("geo_level_2_id")]+1]))
                data$geo_level_1_sd_damage[i] <- as.numeric(unlist(sd_damage_level_1[data[i,c("geo_level_1_id")]+1]))}
    } else {data$geo_level_3_sd_damage[i] <- as.numeric(unlist(sd_damage_level_3[data[i,c("geo_level_3_id")]+1]))
            data$geo_level_2_sd_damage[i] <- as.numeric(unlist(sd_damage_level_2[data[i,c("geo_level_2_id")]+1]))
            data$geo_level_1_sd_damage[i] <- as.numeric(unlist(sd_damage_level_1[data[i,c("geo_level_1_id")]+1]))}
}

In [38]:
for (i in 1:nrow(test)){
    if(is.na(mean_damage_level_3[test[i,c("geo_level_3_id")]+1])){
        if(is.na(mean_damage_level_2[test[i,c("geo_level_2_id")]+1])){
            test$geo_level_3_mean_damage[i] <- as.numeric(unlist(mean_damage_level_1[test[i,c("geo_level_1_id")]+1]))
            test$geo_level_2_mean_damage[i] <- as.numeric(unlist(mean_damage_level_1[test[i,c("geo_level_1_id")]+1]))
            test$geo_level_1_mean_damage[i] <- as.numeric(unlist(mean_damage_level_1[test[i,c("geo_level_1_id")]+1]))
        } else {test$geo_level_3_mean_damage[i] <- as.numeric(unlist(mean_damage_level_2[test[i,c("geo_level_2_id")]+1]))
                test$geo_level_2_mean_damage[i] <- as.numeric(unlist(mean_damage_level_2[test[i,c("geo_level_2_id")]+1]))
                test$geo_level_1_mean_damage[i] <- as.numeric(unlist(mean_damage_level_1[test[i,c("geo_level_1_id")]+1]))}
    } else {test$geo_level_3_mean_damage[i] <- as.numeric(unlist(mean_damage_level_3[test[i,c("geo_level_3_id")]+1]))
            test$geo_level_2_mean_damage[i] <- as.numeric(unlist(mean_damage_level_2[test[i,c("geo_level_2_id")]+1]))
            test$geo_level_1_mean_damage[i] <- as.numeric(unlist(mean_damage_level_1[test[i,c("geo_level_1_id")]+1]))}
}

for (i in 1:nrow(test)){
    if(is.na(sd_damage_level_3[test[i,c("geo_level_3_id")]+1])){
        if(is.na(sd_damage_level_2[test[i,c("geo_level_2_id")]+1])){
            test$geo_level_3_sd_damage[i] <- as.numeric(unlist(sd_damage_level_1[test[i,c("geo_level_1_id")]+1]))
            test$geo_level_2_sd_damage[i] <- as.numeric(unlist(sd_damage_level_1[test[i,c("geo_level_1_id")]+1]))
            test$geo_level_1_sd_damage[i] <- as.numeric(unlist(sd_damage_level_1[test[i,c("geo_level_1_id")]+1]))
        } else {test$geo_level_3_sd_damage[i] <- as.numeric(unlist(sd_damage_level_2[test[i,c("geo_level_2_id")]+1]))
                test$geo_level_2_sd_damage[i] <- as.numeric(unlist(sd_damage_level_2[test[i,c("geo_level_2_id")]+1]))
                test$geo_level_1_sd_damage[i] <- as.numeric(unlist(sd_damage_level_1[test[i,c("geo_level_1_id")]+1]))}
    } else {test$geo_level_3_sd_damage[i] <- as.numeric(unlist(sd_damage_level_3[test[i,c("geo_level_3_id")]+1]))
            test$geo_level_2_sd_damage[i] <- as.numeric(unlist(sd_damage_level_2[test[i,c("geo_level_2_id")]+1]))
            test$geo_level_1_sd_damage[i] <- as.numeric(unlist(sd_damage_level_1[test[i,c("geo_level_1_id")]+1]))}
}

In [185]:
data <- data[,-c(1,2,3,4)]
test <- test[,-c(1,2,3,4)]

In [43]:
data[1:3,]

,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,⋯,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other,damage_grade,geo_level_3_mean_damage,geo_level_2_mean_damage,geo_level_1_mean_damage,geo_level_3_sd_damage,geo_level_2_sd_damage,geo_level_1_sd_damage
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<fct>,<fct>,⋯,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,4,30,266,1224,1,25,5,2,t,r,⋯,0,0,0,2,1.971429,1.984293,2.020477,0.1690309,0.2987067,0.4558226
2,8,17,409,12182,2,0,13,7,t,r,⋯,0,0,0,3,3.000000,2.977444,2.794480,0.0000000,0.1490457,0.4352255
3,12,17,716,7056,2,5,12,6,o,r,⋯,0,0,0,3,3.000000,2.984772,2.794480,0.0000000,0.1227723,0.4352255


In [45]:
data <- data %>% relocate(geo_level_1_mean_damage,.after=geo_level_1_id) %>% 
relocate(geo_level_2_mean_damage,.after=geo_level_2_id) %>% 
relocate(geo_level_3_mean_damage,.after=geo_level_3_id) %>% 
relocate(geo_level_1_sd_damage,.after=geo_level_1_mean_damage) %>%
relocate(geo_level_2_sd_damage,.after=geo_level_2_mean_damage) %>% 
relocate(geo_level_3_sd_damage,.after=geo_level_3_mean_damage)

In [ ]:
test <- test %>% relocate(geo_level_1_mean_damage,.after=geo_level_1_id) %>% 
relocate(geo_level_2_mean_damage,.after=geo_level_2_id) %>% 
relocate(geo_level_3_mean_damage,.after=geo_level_3_id) %>% 
relocate(geo_level_1_sd_damage,.after=geo_level_1_mean_damage) %>%
relocate(geo_level_2_sd_damage,.after=geo_level_2_mean_damage) %>% 
relocate(geo_level_3_sd_damage,.after=geo_level_3_mean_damage)

In [47]:
write.csv(data, file = "data_target_encoding.csv", row.names = FALSE)

In [182]:
#data[c("damage_grade")] <- lapply(data[c("damage_grade")], function(x) as.factor(x))
nzv <- nearZeroVar(data)
data <- data[, -nzv]
test <- test[, -nzv]

In [173]:
dim(data)

[1] 260601     21

In [174]:
data[1:2,]

,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,roof_type,ground_floor_type,other_floor_type,position,⋯,has_superstructure_mud_mortar_stone,has_superstructure_mud_mortar_brick,has_superstructure_cement_mortar_brick,has_superstructure_timber,has_superstructure_bamboo,count_families,has_secondary_use,has_secondary_use_agriculture,damage_grade,geo_level_damage
,<int>,<int>,<int>,<int>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<fct>,<dbl>
1,1,25,5,2,t,r,n,f,j,s,⋯,1,0,0,0,0,0,0,0,2,1.971429
2,2,0,13,7,t,r,n,f,q,s,⋯,1,0,0,0,0,1,0,0,3,3.000000


In [186]:
dummy_data <- dummyVars(" ~ .", data=data)  
data_dummy <- data.frame(predict(dummy_data, newdata = data)) 

In [187]:
cor(data_dummy)

,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition.n,land_surface_condition.o,land_surface_condition.t,foundation_type.h,foundation_type.i,foundation_type.r,⋯,has_superstructure_mud_mortar_stone,has_superstructure_mud_mortar_brick,has_superstructure_cement_mortar_brick,has_superstructure_timber,has_superstructure_bamboo,count_families,has_secondary_use,has_secondary_use_agriculture,damage_grade,geo_level_damage
count_floors_pre_eq,1.000000000,8.666803e-02,0.1010711767,0.7727343845,-0.038525062,-0.0231557691,0.0462183437,-0.0409224848,0.040634019,0.131691925,⋯,-0.027115976,0.2572787019,-0.085820914,-0.0566370921,-0.0704872365,0.0865859897,0.052125461,-0.005293656,0.122308494,0.168440126
age,0.086668031,1.000000e+00,-0.0043225113,0.0610735493,-0.012086677,-0.0156369399,0.0184340079,-0.0049595365,-0.048859993,0.072639710,⋯,0.001321263,0.0795254426,-0.036991916,0.0058554250,-0.0083744692,0.0053092807,-0.008787676,-0.002194027,0.029273302,-0.002786670
area_percentage,0.101071177,-4.322511e-03,1.0000000000,0.1966452848,-0.028422431,0.0005412602,0.0258172259,-0.0089222331,0.301687721,-0.198505861,⋯,-0.225540553,0.0533615320,0.210800360,-0.0539651165,-0.0316412341,0.0886301872,0.122400610,-0.016477524,-0.125220516,-0.078084814
height_percentage,0.772734385,6.107355e-02,0.1966452848,1.0000000000,-0.019977473,-0.0125886951,0.0242399106,-0.0274728807,0.163200663,0.008660952,⋯,-0.106572989,0.2090977687,0.001697751,-0.0524024758,-0.0633418824,0.0643160814,0.091779891,-0.005390015,0.048130024,0.101712559
land_surface_condition.n,-0.038525062,-1.208668e-02,-0.0284224314,-0.0199774726,1.000000000,-0.0721332011,-0.8833954570,-0.0041231877,-0.028119945,0.026637757,⋯,0.074832008,-0.0503819064,-0.054304164,0.0345297494,0.0228840062,-0.0088398241,-0.005813440,0.005457553,0.008529759,0.008013137
land_surface_condition.o,-0.023155769,-1.563694e-02,0.0005412602,-0.0125886951,-0.072133201,1.0000000000,-0.4036853666,-0.0032915220,-0.005485437,-0.002253360,⋯,0.023413596,-0.0356724205,-0.021261808,0.0288783987,0.0209044520,-0.0007053341,0.007868484,0.003258622,0.015077654,0.018382279
land_surface_condition.t,0.046218344,1.843401e-02,0.0258172259,0.0242399106,-0.883395457,-0.4036853666,1.0000000000,0.0053286771,0.028371416,-0.023375733,⋯,-0.079643389,0.0629754548,0.059802434,-0.0452422845,-0.0308132143,0.0084400621,0.001635570,-0.006537215,-0.014908510,-0.015987305
foundation_type.h,-0.040922485,-4.959536e-03,-0.0089222331,-0.0274728807,-0.004123188,-0.0032915220,0.0053286771,1.0000000000,-0.015375858,-0.171987181,⋯,-0.107303112,-0.0105876869,0.005285327,0.0475863740,0.0103474425,-0.0010805502,0.003111238,0.002898566,-0.016201749,-0.010207835
foundation_type.i,0.040634019,-4.885999e-02,0.3016877206,0.1632006631,-0.028119945,-0.0054854365,0.0283714161,-0.0153758579,1.000000000,-0.473285407,⋯,-0.359322680,-0.0439791230,0.246290004,-0.1098119132,-0.0597015353,-0.0269173464,0.167006323,-0.043895800,-0.263900995,-0.140997635
foundation_type.r,0.131691925,7.263971e-02,-0.1985058610,0.0086609521,0.026637757,-0.0022533602,-0.0233757330,-0.1719871811,-0.473285407,1.000000000,⋯,0.538937022,0.0711387338,-0.405372425,-0.1260819270,-0.1360608261,0.0342810159,-0.106780588,0.038260891,0.343355426,0.307699619
